In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Cargar datos
df = pd.read_csv('./data/US_Accidents_2022.csv')
total_records = len(df)

print("="*70)
print(" ANÁLISIS DE CALIDAD DE DATOS")
print("="*70)
print(f"Total de registros: {total_records:,}\n")

In [3]:
# 1. Valores nulos
print("1. VALORES NULOS POR COLUMNA:")
null_counts = df.isnull().sum()
null_percentage = (null_counts / total_records * 100).round(2)
null_summary = pd.DataFrame({
    'Columna': null_counts.index,
    'Nulos': null_counts.values,
    'Porcentaje': null_percentage.values
})
null_summary = null_summary[null_summary['Nulos'] > 0].sort_values('Porcentaje', ascending=False)
print(null_summary.head(15).to_string(index=False))

1. VALORES NULOS POR COLUMNA:
              Columna  Nulos  Porcentaje
              End_Lat 236182       13.40
              End_Lng 236182       13.40
    Precipitation(in)  64475        3.66
        Wind_Chill(F)  54508        3.09
       Wind_Direction  48385        2.75
      Wind_Speed(mph)  48377        2.74
       Visibility(mi)  41953        2.38
          Humidity(%)  41107        2.33
    Weather_Condition  38688        2.20
       Temperature(F)  38718        2.20
         Pressure(in)  33134        1.88
    Weather_Timestamp  29595        1.68
       Sunrise_Sunset  14896        0.85
    Nautical_Twilight  14896        0.85
Astronomical_Twilight  14896        0.85


In [4]:
# 2. Duplicados
duplicates = df.duplicated(subset=['ID']).sum()
duplicate_pct = (duplicates / total_records * 100).round(2)
print(f"\n2. DUPLICADOS:")
print(f"   Registros duplicados: {duplicates:,} ({duplicate_pct}%)")


2. DUPLICADOS:
   Registros duplicados: 0 (0.0%)


In [8]:
# 3. Fechas inválidas
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
df['Year'] = df['Start_Time'].dt.year
invalid_dates = df['Start_Time'].isnull().sum()
non_2022 = ((df['Year'] != 2022) & (df['Year'].notna())).sum()
print(f"\n3. FECHAS:")
print(f"   Fechas no parseables: {invalid_dates:,}")
print(f"   Registros fuera de 2022: {non_2022:,}")


3. FECHAS:
   Fechas no parseables: 0
   Registros fuera de 2022: 0


In [9]:
# 4. Coordenadas fuera de rango USA continental
invalid_coords = ((df['Start_Lat'] < 24) | (df['Start_Lat'] > 49) | 
                  (df['Start_Lng'] < -125) | (df['Start_Lng'] > -66)).sum()
print(f"\n4. COORDENADAS:")
print(f"   Coordenadas fuera de USA continental: {invalid_coords:,}")


4. COORDENADAS:
   Coordenadas fuera de USA continental: 3


In [10]:
# 5. Severidad fuera de rango
invalid_severity = (~df['Severity'].isin([1,2,3,4])).sum()
print(f"\n5. SEVERIDAD:")
print(f"   Valores fuera de rango (1-4): {invalid_severity:,}")



5. SEVERIDAD:
   Valores fuera de rango (1-4): 0


In [12]:
# CALCULAR RUIDO TOTAL
total_noise = duplicates + invalid_dates + invalid_coords + invalid_severity + non_2022
noise_percentage = (total_noise / total_records * 100).round(2)

print(f"\n{'='*70}")
print(f"RESUMEN DE RUIDO:")
print(f"   Total de registros problemáticos: {total_noise:,}")
print(f"   Porcentaje de ruido: {noise_percentage}%")
print(f"{'='*70}")


RESUMEN DE RUIDO:
   Total de registros problemáticos: 3
   Porcentaje de ruido: 0.0%
